# 🛠️ **Functions**

In [1]:
from google.colab import drive
import sys
from skimage import io, exposure, util, filters, morphology
from skimage.morphology import disk
import skimage.io as skio
import ipywidgets as widgets
import skimage
from __future__ import annotations
import ipywidgets as widgets
from IPython.display import display, clear_output
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from PIL import Image, ImageDraw, ImageFont

from pathlib import Path

from tqdm.auto import tqdm
import matplotlib
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np
import pandas as pd
import tifffile as tiff
drive.mount('/content/gdrive')


Mounted at /content/gdrive


In [2]:
DONE_ASCII = r"""
 ____   ___  _   _  _____
|  _ \ / _ \| \ | || ____|
| | | | | | |  \| ||  _|
| |_| | |_| | |\  || |___
|____/ \___/|_| \_||_____|
"""


In [3]:
pip install tifffile

In [4]:
def normalize(img):
  img=(img-np.min(img))/(np.max(img)-np.min(img))
  return img
def normalize_255(img):
  img=((img-np.min(img))/(np.max(img)-np.min(img)))*255
  return img


# 📂**Create the folders**
⚠️ USER INPUT REQUIRED  
*Always run this block*

## Function

In [5]:
def setup_project_paths(
    *,
    drive_root: str = "/content/gdrive/MyDrive/",
    pipeline_folder: str = "pipeline",
    project_subpath: str = "",
    project_name: Optional[str] = None,
    create_project_root: bool = False,
    create_missing_folders: bool = True,
    require_raw_images: bool = True,
) -> Dict[str, str | List[str]]:
    """
    Initialize or load a multiplex imaging project folder structure.

    Returns a dict of normalized paths (strings ending with "/") plus diagnostics.
    """

    drive_root_path = Path(drive_root)
    project_subpath = project_subpath.strip("/")

    if project_name is None:
        entered_subpath = input(
            "Enter subpath under MyDrive (press Enter for none): "
        ).strip()
        if entered_subpath:
            project_subpath = entered_subpath.strip("/")

        project_name = input("Enter project name: ").strip()
        if not project_name:
            raise ValueError("Project name cannot be empty.")

    project_root = drive_root_path / project_subpath / pipeline_folder / project_name

    if project_root.exists() and not project_root.is_dir():
        raise ValueError(f"project_root exists but is not a directory: {project_root}")

    if not project_root.exists():
        if create_project_root:
            project_root.mkdir(parents=True, exist_ok=True)
        else:
            raise FileNotFoundError(f"Project root does not exist: {project_root}")

    path_raw = project_root / "images" / "raw"
    path_mcd = project_root / "images" / "mcd"

    created: List[str] = []
    for p in [path_raw, path_mcd]:
        if create_missing_folders and not p.exists():
            p.mkdir(parents=True, exist_ok=True)
            created.append(str(p))

    warnings: List[str] = []
    if require_raw_images:
        exts = (".tif", ".tiff", ".png", ".ome.tif", ".ome.tiff")
        n_imgs = sum(1 for _ in path_raw.rglob("*") if _.is_file() and _.name.lower().endswith(exts))
        if n_imgs == 0:
            warnings.append(
                f"No raw images found in {path_raw} (expected tif/tiff/png). "
                "If you already ran the export step, check the path."
            )

    def _s(p: Path) -> str:
        return str(p.as_posix().rstrip("/")) + "/"

    return {
        "project_root": _s(project_root),
        "path_raw": _s(path_raw),
        "path_mcd": _s(path_mcd),
        "created_folders": created,
        "warnings": warnings,
    }

## Execution

In [6]:
paths = setup_project_paths()
path = paths["project_root"]
path_raw = paths["path_raw"]
path_mcd = paths["path_mcd"]



Enter subpath under MyDrive (press Enter for none): these
Enter project name: rejection


# 📊 **Images format: MCD --> TIFF**
Execute this block only if you have mcd files

***Put the MCD files in the folder "mcd" (project/images/mcd)***

### Functions

In [ ]:
%pip install readimc
from readimc import MCDFile, TXTFile

In [ ]:
def convert_mcd_tiff(path_mcd,roi_exclude=[],marker_exclude=[],path="./images/raw_images"):
    if os.path.isdir(path)==False:
        os.mkdir(path)
    for file in os.listdir(path_mcd):
        print("   ** "+file+" **")
        with MCDFile(path_mcd+"/"+file) as f:
            slide = f.slides[0]
            panorama = slide.panoramas[0]
            for acq in range(len(slide.acquisitions)):
                acquisition = slide.acquisitions[acq]
                roi=acquisition.description
                if roi not in roi_exclude:
                    print("     ROI: "+roi)
                    if os.path.isdir(path+"/"+acquisition.description)==False:
                         os.mkdir(path+"/"+acquisition.description)
                    try:
                        img = f.read_acquisition(acquisition)
                        list_target=acquisition.channel_labels
                        dico_target={v:i for i,v in enumerate(list_target)}
                        for i in range(len(list_target)):
                            if list_target[i] not in marker_exclude:
                                img_marker=img[dico_target[list_target[i]],:,:]
                                cv2.imwrite(path+"/"+acquisition.description+"/"+list_target[i]+".tiff",img_marker)

                    except:
                        print("     Erreure: "+roi)
    print("✅ All the images are available in the folder: "+path)




### Execution

In [ ]:
convert_mcd_tiff(path_mcd=path_mcd,path=path_raw)


   ** Lame_1.mcd **
     ROI: 19U07351a
     ROI: 19U07351b
     ROI: 19U07351c
     ROI: 19U07351d
     ROI: 19U07351e
     ROI: 19U07351f
     ROI: 19U07351g
     ROI: 20U01680a
     ROI: 20U01680b
     ROI: 20U01680c
     ROI: 20U01680d
     ROI: 20U01680e
     ROI: 20U04212a
     ROI: 20U04212b
     ROI: 20U04212c
     ROI: 20U04212d
     ROI: 20U04212e
     ROI: 20U04212f
     ROI: 20U06040a
     ROI: 20U06040b
     ROI: test
     ROI: test2
     ROI: test3
     ROI: test4
     ROI: 19U07351bb
     ROI: ROI_029
✅ All the images are available in the folder: /content/gdrive/MyDrive/these/pipeline/VAA//images/raw/


# 📊 **Standart pre-processing**
⚠️ USER INPUT REQUIRED  
  
Image processing to enable better viewing
Threshold on the lower and upper quantil + arcsinh with cofactor + CLAHE (Contrast Limited Adaptive Histogram Equalization)

In [ ]:
path_img=path+"/images/"
path_raw=path+"/images/raw/"
path_img_preprocessed1=path+"/images/img_processing_1/"
path_img_preprocessed1_marker=path_img_preprocessed1+"Marker/"
path_img_preprocessed1_biopsies=path_img_preprocessed1+"Biopsies/"

In [ ]:
if os.path.isdir(path_img_preprocessed1)==False:
  os.mkdir(path_img_preprocessed1)
  print("✅ Folder for images preprocessed created")
if os.path.isdir(path_img_preprocessed1_biopsies)==False:
  os.mkdir(path_img_preprocessed1_biopsies)
  print("✅ Folder for images preprocessed created")
if os.path.isdir(path_img_preprocessed1_marker)==False:
  os.mkdir(path_img_preprocessed1_marker)

✅ Folder for images preprocessed created
✅ Folder for images preprocessed created


### Functions

In [ ]:

def imc_display_transform(
    img,               # 2D numpy array, IMC channel (float32/float64/uint16)
    low_q=1.0,         # lower quantile (background)
    high_q=99.9,       # upper quantile (clip)
    cofactor=5.0,      # arcsinh cofactor
    clahe=False,       # True to apply CLAHE (for visualization only)
    clahe_kernel=128,  # CLAHE tile size
    clahe_clip=0.01    # CLAHE clip limit
):
    x = img.astype(np.float32)

    # 1) Background subtraction (based on lower quantile)
    bg = np.quantile(x, low_q / 100.0)
    x = np.clip(x - bg, 0, None)

    # 2) Robust upper clipping
    hi = np.quantile(x, high_q / 100.0) if np.isfinite(x).all() else np.percentile(x, high_q)
    if hi > 0:
        x = np.clip(x, 0, hi)

    # 3) Arcsinh scaling (CyTOF-style)
    x = np.arcsinh(x / max(cofactor, 1e-6))

    # 4) Rescale to 0–1 range
    x = x - x.min()
    if x.max() > 0:
        x = x / x.max()

    # 5) Optional CLAHE (for display enhancement)
    # skimage CLAHE expects 8-bit or float [0,1]
    if clahe:
        x = exposure.equalize_adapthist(x, kernel_size=clahe_kernel, clip_limit=clahe_clip)

    return x  # float [0,1] ready for display or 8-bit conversion

# --- Example usage per channel ---
# raw = io.imread("path/my_channel.tiff")  # 16/32-bit
# disp = imc_display_transform(raw, low_q=1.0, high_q=99.9, cofactor=5.0, clahe=False)
# io.imsave("channel_display.png", util.img_as_ubyte(disp))


In [ ]:
# =========================================================
# IMC display preprocessing (imc_display_transform) — interactive Colab UI
# - Numeric parameter inputs (no sliders)
# - Choose ROI + marker for live preview
# - KEEP same marker when ROI changes (by stem)
# - Real-time preview update
# - Button: save ALL images
# - Button: save SELECTED MARKER across ALL ROIs
#
# Outputs:
#   1) by biopsy:  path_img_preprocessed1_biopsies/<ROI>/<marker>.png
#   2) by marker:  path_img_preprocessed1_marker/<marker>/<ROI>.png
# =========================================================





IMG_EXTS = (".tif", ".tiff", ".png", ".jpg", ".jpeg")


# -----------------------------
# Helpers
# -----------------------------
def _list_rois_and_markers(path_raw: str) -> tuple[list[str], dict[str, list[str]]]:
    rois: list[str] = []
    roi_to_markers: dict[str, list[str]] = {}
    for roi in sorted(os.listdir(path_raw)):
        roi_dir = os.path.join(path_raw, roi)
        if not os.path.isdir(roi_dir):
            continue
        rois.append(roi)
        markers = []
        for f in sorted(os.listdir(roi_dir)):
            p = os.path.join(roi_dir, f)
            if os.path.isfile(p) and f.lower().endswith(IMG_EXTS):
                markers.append(f)
        roi_to_markers[roi] = markers
    return rois, roi_to_markers


def _read_image(path: str) -> np.ndarray:
    # This is your input format (IMC raw marker channels)
    return tiff.imread(path)


def _safe_imshow(img_u8: np.ndarray, title: str = "") -> None:
    plt.figure(figsize=(7, 7))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.imshow(img_u8, interpolation="nearest")
    plt.show()


def run_interactive_imc_display_preprocessing(
    *,
    path_raw: str,
    path_img_preprocessed1_biopsies: str,
    path_img_preprocessed1_marker: str,
    default_low_q: float = 1.0,
    default_high_q: float = 99.0,
    default_cofactor: float = 5.0,
):
    """
    Requires existing functions:
      - imc_display_transform(img, low_q, high_q, cofactor) -> float/array
      - normalize_255(x) -> uint8
    """
    rois, roi_to_markers = _list_rois_and_markers(path_raw)
    if not rois:
        raise ValueError(f"No ROI folders found in: {path_raw}")

    # ---------- widgets ----------
    header = widgets.HTML("<h3 style='margin:0 0 10px 0;'>IMC preprocessing — imc_display_transform</h3>")

    roi_dd = widgets.Dropdown(
        options=rois, value=rois[0], description="ROI:",
        layout=widgets.Layout(width="520px"), style={"description_width": "80px"}
    )

    first_roi_markers = roi_to_markers.get(rois[0], [])
    marker_dd = widgets.Dropdown(
        options=first_roi_markers,
        value=first_roi_markers[0] if first_roi_markers else None,
        description="Marker:",
        layout=widgets.Layout(width="520px"), style={"description_width": "80px"}
    )

    lowq_w = widgets.FloatText(
        value=float(default_low_q), description="low_q:",
        layout=widgets.Layout(width="520px"), style={"description_width": "80px"}
    )
    highq_w = widgets.FloatText(
        value=float(default_high_q), description="high_q:",
        layout=widgets.Layout(width="520px"), style={"description_width": "80px"}
    )
    cof_w = widgets.FloatText(
        value=float(default_cofactor), description="cofactor:",
        layout=widgets.Layout(width="520px"), style={"description_width": "80px"}
    )

    btn_update = widgets.Button(description="Run / Update preview", button_style="primary", icon="play")
    btn_save_all = widgets.Button(description="Save ALL images", button_style="success", icon="download")
    btn_save_marker = widgets.Button(description="Save selected marker (ALL ROIs)", button_style="warning", icon="save")

    qc_mode = widgets.Dropdown(
        options=[("No QC plots", "none"), ("Every N images", "every_n"), ("First K images", "first_k")],
        value="first_k",
        description="QC:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "80px"},
    )
    qc_every_n = widgets.IntText(value=200, description="Every N:", layout=widgets.Layout(width="260px"),
                                 style={"description_width": "80px"})
    qc_first_k = widgets.IntText(value=10, description="First K:", layout=widgets.Layout(width="260px"),
                                 style={"description_width": "80px"})

    status = widgets.HTML("")
    out = widgets.Output()
    log = widgets.Output()

    # ---------- logic ----------
    def _validate_params():
        low_q = float(lowq_w.value)
        high_q = float(highq_w.value)
        cof = float(cof_w.value)

        if not (0 <= low_q < high_q <= 100):
            raise ValueError("Require 0 <= low_q < high_q <= 100")
        if cof == 0:
            raise ValueError("cofactor must be != 0")
        return low_q, high_q, cof

    def _should_qc(i: int, mode: str, every_n: int, first_k: int) -> bool:
        if mode == "none":
            return False
        if mode == "every_n":
            n = max(int(every_n), 1)
            return (i % n) == 0
        if mode == "first_k":
            k = max(int(first_k), 0)
            return i < k
        return False

    def _update_marker_options(*_):
        roi = roi_dd.value
        opts = roi_to_markers.get(roi, [])

        current = marker_dd.value
        current_stem = Path(current).stem if current else None

        marker_dd.options = opts
        if not opts:
            marker_dd.value = None
            return

        if current_stem is not None:
            for f in opts:
                if Path(f).stem == current_stem:
                    marker_dd.value = f
                    return

        if current in opts:
            marker_dd.value = current
            return

        marker_dd.value = opts[0]

    def _preview(*_):
        with out:
            clear_output(wait=True)

            roi = roi_dd.value
            marker = marker_dd.value
            if marker is None:
                status.value = "<b style='color:#b00020'>No marker available for this ROI.</b>"
                return

            try:
                low_q, high_q, cof = _validate_params()
            except Exception as e:
                status.value = f"<b style='color:#b00020'>Error:</b> {e}"
                return

            in_path = os.path.join(path_raw, roi, marker)
            if not os.path.exists(in_path):
                status.value = f"<b style='color:#b00020'>Missing file:</b> {in_path}"
                return

            img = _read_image(in_path)
            img_disp = imc_display_transform(img, low_q=low_q, high_q=high_q, cofactor=cof)
            img_u8 = normalize_255(img_disp)

            _safe_imshow(img_u8, title=f"{roi} | {Path(marker).stem} | low={low_q} high={high_q} cof={cof}")

            status.value = (
                f"<b>Preview:</b> ROI={roi} | marker={Path(marker).stem} "
                f"| low_q={low_q} | high_q={high_q} | cofactor={cof}"
            )

    def _save_one(roi: str, marker_filename: str, low_q: float, high_q: float, cof: float):
        roi_in = os.path.join(path_raw, roi)
        marker_path = os.path.join(roi_in, marker_filename)

        marker_name = Path(marker_filename).stem
        out_biopsy_dir = os.path.join(path_img_preprocessed1_biopsies, roi)
        os.makedirs(out_biopsy_dir, exist_ok=True)

        out_marker_dir = os.path.join(path_img_preprocessed1_marker, marker_name)
        os.makedirs(out_marker_dir, exist_ok=True)

        img = _read_image(marker_path)
        img_disp = imc_display_transform(img, low_q=low_q, high_q=high_q, cofactor=cof)
        img_u8 = normalize_255(img_disp)

        out1 = os.path.join(out_biopsy_dir, marker_name + ".png")
        out2 = os.path.join(out_marker_dir, roi + ".png")
        cv2.imwrite(out1, img_u8)
        cv2.imwrite(out2, img_u8)

        return marker_name, img_u8

    def _save_all(_):
        with log:
            clear_output(wait=True)
            try:
                low_q, high_q, cof = _validate_params()
            except Exception as e:
                print(f"[ERROR] {e}")
                return

            total = sum(len(roi_to_markers.get(roi, [])) for roi in rois)
            os.makedirs(path_img_preprocessed1_biopsies, exist_ok=True)
            os.makedirs(path_img_preprocessed1_marker, exist_ok=True)

            mode = str(qc_mode.value)
            every_n = int(qc_every_n.value)
            first_k = int(qc_first_k.value)

            done = 0
            failed = 0

            with tqdm(total=total, desc="IMC preprocessing (ALL)", unit="img", ncols=100) as pbar:
                for roi in rois:
                    for marker in roi_to_markers.get(roi, []):
                        try:
                            marker_name, img_u8 = _save_one(roi, marker, low_q, high_q, cof)

                            if _should_qc(done, mode, every_n, first_k):
                                with out:
                                    clear_output(wait=True)
                                    _safe_imshow(img_u8, title=f"QC {done+1}/{total} | {roi} | {marker_name}")

                            done += 1
                        except Exception as e:
                            failed += 1
                            print(f"[WARN] Failed ROI={roi} marker={marker}: {e}")
                        pbar.update(1)

            print(f"\n✅ By biopsies: {path_img_preprocessed1_biopsies}")
            print(f"✅ By marker:  {path_img_preprocessed1_marker}")
            print(f"Done={done} | Failed={failed} | low_q={low_q} high_q={high_q} cofactor={cof}")

    def _save_selected_marker(_):
        with log:
            clear_output(wait=True)

            marker_filename = marker_dd.value
            if marker_filename is None:
                print("[ERROR] No marker selected.")
                return
            marker_stem = Path(marker_filename).stem

            try:
                low_q, high_q, cof = _validate_params()
            except Exception as e:
                print(f"[ERROR] {e}")
                return

            def _find_marker_in_roi(roi: str, stem: str) -> str | None:
                for f in roi_to_markers.get(roi, []):
                    if Path(f).stem == stem:
                        return f
                return None

            total = len(rois)
            mode = str(qc_mode.value)
            every_n = int(qc_every_n.value)
            first_k = int(qc_first_k.value)

            done = 0
            skipped = 0
            failed = 0

            with tqdm(total=total, desc=f"Save marker '{marker_stem}' (ALL ROIs)", unit="ROI", ncols=100) as pbar:
                for i, roi in enumerate(rois):
                    f = _find_marker_in_roi(roi, marker_stem)
                    if f is None:
                        skipped += 1
                        pbar.update(1)
                        continue

                    try:
                        marker_name, img_u8 = _save_one(roi, f, low_q, high_q, cof)

                        if _should_qc(done, mode, every_n, first_k):
                            with out:
                                clear_output(wait=True)
                                _safe_imshow(img_u8, title=f"QC {done+1}/{total} | {roi} | {marker_name}")

                        done += 1
                    except Exception as e:
                        failed += 1
                        print(f"[WARN] Failed ROI={roi} marker={f}: {e}")

                    pbar.update(1)

            print(f"\n✅ Saved selected marker across ROIs: {marker_stem}")
            print(f"✅ By biopsies: {path_img_preprocessed1_biopsies}")
            print(f"✅ By marker:  {path_img_preprocessed1_marker}")
            print(f"Done={done} | Skipped(missing marker)={skipped} | Failed={failed} | low_q={low_q} high_q={high_q} cof={cof}")

    # ---------- wiring ----------
    roi_dd.observe(_update_marker_options, names="value")

    btn_update.on_click(lambda _: _preview())
    btn_save_all.on_click(_save_all)
    btn_save_marker.on_click(_save_selected_marker)

    # live preview on change
    marker_dd.observe(_preview, names="value")
    lowq_w.observe(_preview, names="value")
    highq_w.observe(_preview, names="value")
    cof_w.observe(_preview, names="value")

    # layout
    display(widgets.HBox([
        widgets.VBox([
            header,
            roi_dd,
            marker_dd,
            widgets.HTML("<b>Parameters</b>"),
            lowq_w,
            highq_w,
            cof_w,
            widgets.HBox([btn_update, btn_save_all, btn_save_marker]),
            widgets.HTML("<b>Batch QC (avoid flooding output)</b>"),
            qc_mode,
            widgets.HBox([qc_every_n, qc_first_k]),
            status,
            log
        ], layout=widgets.Layout(width="560px")),
        widgets.VBox([
            widgets.HTML("<b>Preview</b>"),
            out
        ], layout=widgets.Layout(width="700px"))
    ], layout=widgets.Layout(gap="18px")))

    _preview()



### Execution
⚠️ USER INPUT REQUIRED  

In [ ]:
run_interactive_imc_display_preprocessing(
    path_raw=path_raw,
    path_img_preprocessed1_biopsies=path_img_preprocessed1_biopsies,
    path_img_preprocessed1_marker=path_img_preprocessed1_marker,
    #default_low_q=low_q,
    #default_high_q=high_q,
    #default_cofactor=cofactor,
)

# 📊 **Pipeline of Pre-processing**
⚠️ USER INPUT REQUIRED  
Generating pre-processed images with less noise ordered by biopsy
Pseudo code of the pre-processing:


*   Arcsinh_image=Arcsinh(image*cofactor) --> Increase variance between low and high pixel intensity
*   Blur_image=Blur(image) --> Remove noise between real cells

*   Standardized_image=Stardadization(Blur_image) --> Facilitate the thresholding
*   Final_image=threshold(Standardized_image) --> If the pixel intensity of the stabdardized image is < threshold , it's replaced by 0 , else we keep the Arcsinh_image values

**How to tune parameters from the visual output:**

* **Cofactor controls** compression strength: increase cofactor if the image
looks too “flat/dim” (signal overly compressed), decrease cofactor if bright structures saturate and dominate contrast.

* **Threshold** controls background removal: increase threshold if you still see widespread haze/background, decrease threshold if true biological structures disappear or become fragmented.

* **Kernel** controls smoothing/denoising: increase kernel if the result is speckled/noisy, decrease kernel if edges become blurred or fine structures (thin vessels, small cells) are washed out.

A good practice is to tune on a representative ROI/marker (or a few), verify that background is suppressed without erasing dim true signal, and then lock parameters and run the batch export with periodic QC plots to ensure consistency.



In [ ]:
path_img=path+"/images/"
path_raw=path+"/images/raw/"
path_img_preprocessed2=path+"/images/img_processing_2/"
path_img_preprocessed2_marker=path_img_preprocessed2+"Marker/"
path_img_preprocessed2_biopsies=path_img_preprocessed2+"Biopsies/"

In [ ]:
if os.path.isdir(path_img_preprocessed2)==False:
  os.mkdir(path_img_preprocessed2)
  print("✅ Folder for images preprocessed created")
if os.path.isdir(path_img_preprocessed2_biopsies)==False:
  os.mkdir(path_img_preprocessed2_biopsies)
  print("✅ Folder for images preprocessed created")
if os.path.isdir(path_img_preprocessed2_marker)==False:
  os.mkdir(path_img_preprocessed2_marker)

## Functions

In [ ]:

def remove_image_extension(filename: str) -> str:
    """
    Remove image extension from filename, robust to multiple extensions
    (e.g. .ome.tif, .tar.gz-like patterns for images).
    """
    name = os.path.basename(filename)

    image_exts = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp", ".ome"}

    while True:
        root, ext = os.path.splitext(name)
        if ext.lower() in image_exts:
            name = root
        else:
            break

    return name


In [ ]:

IMG_EXTS = (".tif", ".tiff", ".png", ".jpg", ".jpeg")

import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# Helpers
# -----------------------------
def _list_rois_and_markers(path_raw: str) -> tuple[list[str], dict[str, list[str]]]:
    path_raw = str(path_raw)
    rois: list[str] = []
    roi_to_markers: dict[str, list[str]] = {}

    for roi in sorted(os.listdir(path_raw)):
        roi_dir = os.path.join(path_raw, roi)
        if not os.path.isdir(roi_dir):
            continue

        rois.append(roi)
        markers: list[str] = []
        for f in sorted(os.listdir(roi_dir)):
            p = os.path.join(roi_dir, f)
            if os.path.isfile(p) and f.lower().endswith(IMG_EXTS):
                markers.append(f)

        roi_to_markers[roi] = markers

    return rois, roi_to_markers


def _safe_imshow(img_u8: np.ndarray, title: str = "") -> None:
    plt.figure(figsize=(7, 7))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.imshow(img_u8, interpolation="nearest")
    plt.show()


def _read_image_full(path: str) -> np.ndarray:
    return tiff.imread(path)


def _read_image_preview(
    path: str,
    *,
    crop_mode: str = "auto",          # "auto" | "center" | "random" | "full"
    crop_size: int = 1024,            # square crop size (px)
    downsample: int = 1,              # integer stride for preview
    auto_threshold_px: int = 3000,    # if max(H,W) > this and crop_mode="auto" => crop
    rng: np.random.Generator | None = None,
) -> np.ndarray:
    """
    Preview reader:
      - For TIFF: uses tifffile.memmap to avoid loading whole image when cropping.
      - For other formats: falls back to full read (usually fine for PNG/JPEG).

    Returns a 2D numpy array (view/crop) (may still be large if crop_mode="full").
    """
    rng = rng or np.random.default_rng(0)

    ext = Path(path).suffix.lower()
    ds = max(int(downsample), 1)
    cs = max(int(crop_size), 64)

    def _resolve_mode(H: int, W: int) -> str:
        mode = str(crop_mode).lower().strip()
        if mode == "auto":
            mode = "center" if max(H, W) > int(auto_threshold_px) else "full"
        return mode

    if ext in (".tif", ".tiff"):
        arr = tiff.memmap(path)  # lazy for huge TIFF
        if arr.ndim > 2:
            arr = arr[..., 0]
        H, W = arr.shape[:2]

        mode = _resolve_mode(H, W)

        if mode in ("center", "random"):
            ph = min(cs, H)
            pw = min(cs, W)
            if mode == "center":
                y0 = max(0, (H - ph) // 2)
                x0 = max(0, (W - pw) // 2)
            else:
                y0 = int(rng.integers(0, max(1, H - ph + 1)))
                x0 = int(rng.integers(0, max(1, W - pw + 1)))

            view = np.asarray(arr[y0:y0 + ph:ds, x0:x0 + pw:ds], dtype=arr.dtype)
            return view

        # full (but downsampled)
        return np.asarray(arr[::ds, ::ds], dtype=arr.dtype)

    # non-TIFF: load normally then crop/downsample
    arr = tiff.imread(path)
    if arr.ndim > 2:
        arr = arr[..., 0]
    H, W = arr.shape[:2]

    mode = _resolve_mode(H, W)

    if mode in ("center", "random"):
        ph = min(cs, H)
        pw = min(cs, W)
        if mode == "center":
            y0 = max(0, (H - ph) // 2)
            x0 = max(0, (W - pw) // 2)
        else:
            y0 = int(rng.integers(0, max(1, H - ph + 1)))
            x0 = int(rng.integers(0, max(1, W - pw + 1)))
        return arr[y0:y0 + ph:ds, x0:x0 + pw:ds]

    return arr[::ds, ::ds]


def _tqdm_bar(total: int, desc: str, unit: str = "img"):
    """
    Single-line, clean tqdm bar for notebooks/Colab.
    Use tqdm.write() for logs/warnings to avoid breaking the bar line.
    """
    return tqdm(
        total=total,
        desc=desc,
        unit=unit,
        dynamic_ncols=True,
        leave=True,
        file=sys.stdout,
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
    )


# -----------------------------
# Main interactive UI
# -----------------------------
def run_interactive_imc_preprocessing(
    *,
    path_raw: str,
    path_img_preprocessed2_biopsies: str,
    path_img_preprocessed2_marker: str,
    default_threshold: float = 1.0,
    default_cofactor: float = 5.0,
    default_kernel: int = 3,
):
    """
    Requires existing functions in your environment:
      - arcsinh_std_thresh(img, threshold, cofactor, kernel)
      - normalize_255(img_proc) -> uint8
      - remove_image_extension(filename) -> stem without extension
    """
    rois, roi_to_markers = _list_rois_and_markers(path_raw)
    if not rois:
        raise ValueError(f"No ROI folders found in: {path_raw}")

    rng = np.random.default_rng(0)

    # ---------- widgets ----------
    header = widgets.HTML("<h3 style='margin:0 0 10px 0;'>IMC preprocessing — arcsinh_std_thresh</h3>")

    roi_dd = widgets.Dropdown(
        options=rois,
        value=rois[0],
        description="ROI:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "80px"},
    )

    first_roi_markers = roi_to_markers.get(rois[0], [])
    marker_dd = widgets.Dropdown(
        options=first_roi_markers,
        value=first_roi_markers[0] if first_roi_markers else None,
        description="Marker:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "80px"},
    )

    thr_w = widgets.FloatText(
        value=float(default_threshold),
        description="threshold:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "80px"},
    )
    cof_w = widgets.FloatText(
        value=float(default_cofactor),
        description="cofactor:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "80px"},
    )
    ker_w = widgets.IntText(
        value=int(default_kernel),
        description="kernel:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "80px"},
    )

    # preview controls
    preview_mode_w = widgets.Dropdown(
        options=[
            ("Auto (crop if large)", "auto"),
            ("Center crop", "center"),
            ("Random crop", "random"),
            ("Full (downsample)", "full"),
        ],
        value="auto",
        description="Preview:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "80px"},
    )
    crop_size_w = widgets.IntText(
        value=1024,
        description="crop_px:",
        layout=widgets.Layout(width="260px"),
        style={"description_width": "80px"},
    )
    downsample_w = widgets.IntText(
        value=1,
        description="downsample:",
        layout=widgets.Layout(width="260px"),
        style={"description_width": "80px"},
    )
    auto_thr_w = widgets.IntText(
        value=3000,
        description="auto_thr:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "80px"},
    )

    btn_update = widgets.Button(description="Run / Update preview", button_style="primary", icon="play")
    btn_save_all = widgets.Button(description="Save ALL images", button_style="success", icon="download")
    btn_save_marker = widgets.Button(description="Save selected marker (ALL ROIs)", button_style="warning", icon="save")

    qc_mode = widgets.Dropdown(
        options=[("No QC plots", "none"), ("Every N images", "every_n"), ("First K images", "first_k")],
        value="first_k",
        description="QC:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "80px"},
    )
    qc_every_n = widgets.IntText(
        value=200,
        description="Every N:",
        layout=widgets.Layout(width="260px"),
        style={"description_width": "80px"},
    )
    qc_first_k = widgets.IntText(
        value=10,
        description="First K:",
        layout=widgets.Layout(width="260px"),
        style={"description_width": "80px"},
    )

    status = widgets.HTML("")
    out = widgets.Output()
    log = widgets.Output()

    # ---------- logic ----------
    def _validate_params():
        thr = float(thr_w.value)
        cof = float(cof_w.value)
        ker = int(ker_w.value)
        if cof == 0:
            raise ValueError("cofactor must be != 0")
        if ker < 0:
            raise ValueError("kernel must be >= 0")
        return thr, cof, ker

    def _validate_preview():
        cs = int(crop_size_w.value)
        ds = int(downsample_w.value)
        ath = int(auto_thr_w.value)
        if cs <= 0:
            raise ValueError("crop_px must be > 0")
        if ds <= 0:
            raise ValueError("downsample must be > 0")
        if ath <= 0:
            raise ValueError("auto_thr must be > 0")
        return str(preview_mode_w.value), cs, ds, ath

    def _should_qc(i: int, mode: str, every_n: int, first_k: int) -> bool:
        if mode == "none":
            return False
        if mode == "every_n":
            n = max(int(every_n), 1)
            return (i % n) == 0
        if mode == "first_k":
            k = max(int(first_k), 0)
            return i < k
        return False

    def _update_marker_options(*_):
        roi = roi_dd.value
        opts = roi_to_markers.get(roi, [])

        current = marker_dd.value
        current_stem = Path(current).stem if current else None

        marker_dd.options = opts

        if not opts:
            marker_dd.value = None
            return

        if current_stem is not None:
            for f in opts:
                if Path(f).stem == current_stem:
                    marker_dd.value = f
                    return

        if current in opts:
            marker_dd.value = current
            return

        marker_dd.value = opts[0]

    def _preview(*_):
        with out:
            clear_output(wait=True)

            roi = roi_dd.value
            marker = marker_dd.value
            if marker is None:
                status.value = "<b style='color:#b00020'>No marker available for this ROI.</b>"
                return

            try:
                thr, cof, ker = _validate_params()
                pmode, cs, ds, ath = _validate_preview()
            except Exception as e:
                status.value = f"<b style='color:#b00020'>Error:</b> {e}"
                return

            in_path = os.path.join(path_raw, roi, marker)
            if not os.path.exists(in_path):
                status.value = f"<b style='color:#b00020'>Missing file:</b> {in_path}"
                return

            img_prev = _read_image_preview(
                in_path,
                crop_mode=pmode,
                crop_size=cs,
                downsample=ds,
                auto_threshold_px=ath,
                rng=rng,
            )

            img_proc = arcsinh_std_thresh(img_prev, thr, cof, ker)
            img_u8 = normalize_255(img_proc)

            _safe_imshow(
                img_u8,
                title=f"{roi} | {Path(marker).stem} | thr={thr} cof={cof} ker={ker} | "
                      f"preview={pmode} crop={cs} ds={ds}",
            )

            status.value = (
                f"<b>Preview:</b> ROI={roi} | marker={Path(marker).stem} | "
                f"threshold={thr} | cofactor={cof} | kernel={ker} | "
                f"preview={pmode} | crop_px={cs} | downsample={ds}"
            )

    def _save_one(roi: str, marker_filename: str, thr: float, cof: float, ker: int):
        roi_in = os.path.join(path_raw, roi)
        marker_path = os.path.join(roi_in, marker_filename)

        marker_name = Path(marker_filename).stem

        out_biopsy_dir = os.path.join(path_img_preprocessed2_biopsies, roi)
        os.makedirs(out_biopsy_dir, exist_ok=True)

        out_marker_dir = os.path.join(path_img_preprocessed2_marker, marker_name)
        os.makedirs(out_marker_dir, exist_ok=True)

        # full resolution for saving
        img = _read_image_full(marker_path)
        img_proc = arcsinh_std_thresh(img, thr, cof, ker)
        img_u8 = normalize_255(img_proc)

        out1 = os.path.join(out_biopsy_dir, remove_image_extension(marker_filename) + ".png")
        out2 = os.path.join(out_marker_dir, roi + ".png")

        ok1 = cv2.imwrite(out1, img_u8)
        ok2 = cv2.imwrite(out2, img_u8)
        if not (ok1 and ok2):
            raise IOError(f"cv2.imwrite failed for ROI={roi}, marker={marker_filename}")

        return marker_name, img_u8

    def _save_all(_):
        with log:
            clear_output(wait=True)

            try:
                thr, cof, ker = _validate_params()
            except Exception as e:
                print(f"[ERROR] {e}")
                return

            total = sum(len(roi_to_markers.get(roi, [])) for roi in rois)
            if total == 0:
                print("[ERROR] No images found to process.")
                return

            os.makedirs(path_img_preprocessed2_biopsies, exist_ok=True)
            os.makedirs(path_img_preprocessed2_marker, exist_ok=True)

            mode = str(qc_mode.value)
            every_n = int(qc_every_n.value)
            first_k = int(qc_first_k.value)

            done = 0
            failed = 0

            with _tqdm_bar(total, "IMC preprocessing arcsinh (ALL)", unit="img") as pbar:
                for roi in rois:
                    for marker in roi_to_markers.get(roi, []):
                        try:
                            marker_name, img_u8 = _save_one(roi, marker, thr, cof, ker)

                            if _should_qc(done, mode, every_n, first_k):
                                with out:
                                    clear_output(wait=True)
                                    _safe_imshow(img_u8, title=f"QC {done+1}/{total} | {roi} | {marker_name}")

                            done += 1

                        except Exception as e:
                            failed += 1
                            tqdm.write(f"[WARN] Failed ROI={roi} marker={marker}: {e}")

                        pbar.update(1)

            tqdm.write(f"\n✅ By biopsies: {path_img_preprocessed2_biopsies}")
            tqdm.write(f"✅ By marker:  {path_img_preprocessed2_marker}")
            tqdm.write(f"Done={done} | Failed={failed} | threshold={thr} cofactor={cof} kernel={ker}")

    def _save_selected_marker(_):
        with log:
            clear_output(wait=True)

            marker_filename = marker_dd.value
            if marker_filename is None:
                print("[ERROR] No marker selected.")
                return

            marker_stem = Path(marker_filename).stem

            try:
                thr, cof, ker = _validate_params()
            except Exception as e:
                print(f"[ERROR] {e}")
                return

            def _find_marker_in_roi(roi: str, stem: str) -> str | None:
                for f in roi_to_markers.get(roi, []):
                    if Path(f).stem == stem:
                        return f
                return None

            total = len(rois)
            mode = str(qc_mode.value)
            every_n = int(qc_every_n.value)
            first_k = int(qc_first_k.value)

            done = 0
            skipped = 0
            failed = 0

            with _tqdm_bar(total, f"Save marker '{marker_stem}' (ALL ROIs)", unit="ROI") as pbar:
                for roi in rois:
                    f = _find_marker_in_roi(roi, marker_stem)
                    if f is None:
                        skipped += 1
                        pbar.update(1)
                        continue

                    try:
                        marker_name, img_u8 = _save_one(roi, f, thr, cof, ker)

                        if _should_qc(done, mode, every_n, first_k):
                            with out:
                                clear_output(wait=True)
                                _safe_imshow(img_u8, title=f"QC {done+1}/{total} | {roi} | {marker_name}")

                        done += 1

                    except Exception as e:
                        failed += 1
                        tqdm.write(f"[WARN] Failed ROI={roi} marker={f}: {e}")

                    pbar.update(1)

            tqdm.write(f"\n✅ Saved selected marker across ROIs: {marker_stem}")
            tqdm.write(f"✅ By biopsies: {path_img_preprocessed2_biopsies}")
            tqdm.write(f"✅ By marker:  {path_img_preprocessed2_marker}")
            tqdm.write(
                f"Done={done} | Skipped(missing marker)={skipped} | Failed={failed} | "
                f"thr={thr} cof={cof} ker={ker}"
            )

    # ---------- wiring ----------
    roi_dd.observe(_update_marker_options, names="value")

    btn_update.on_click(lambda _: _preview())
    btn_save_all.on_click(_save_all)
    btn_save_marker.on_click(_save_selected_marker)

    # live preview on change
    marker_dd.observe(_preview, names="value")
    thr_w.observe(_preview, names="value")
    cof_w.observe(_preview, names="value")
    ker_w.observe(_preview, names="value")
    preview_mode_w.observe(_preview, names="value")
    crop_size_w.observe(_preview, names="value")
    downsample_w.observe(_preview, names="value")
    auto_thr_w.observe(_preview, names="value")

    # ---------- layout ----------
    display(
        widgets.HBox(
            [
                widgets.VBox(
                    [
                        header,
                        roi_dd,
                        marker_dd,
                        widgets.HTML("<b>Parameters</b>"),
                        thr_w,
                        cof_w,
                        ker_w,
                        widgets.HBox([btn_update, btn_save_all, btn_save_marker]),
                        widgets.HTML("<b>Preview controls (for large biopsies)</b>"),
                        preview_mode_w,
                        widgets.HBox([crop_size_w, downsample_w]),
                        auto_thr_w,
                        widgets.HTML("<b>Batch QC (avoid flooding output)</b>"),
                        qc_mode,
                        widgets.HBox([qc_every_n, qc_first_k]),
                        status,
                        log,
                    ],
                    layout=widgets.Layout(width="560px"),
                ),
                widgets.VBox(
                    [
                        widgets.HTML("<b>Preview</b>"),
                        out,
                    ],
                    layout=widgets.Layout(width="700px"),
                ),
            ],
            layout=widgets.Layout(gap="18px"),
        )
    )

    _preview()

In [ ]:
def arcsinh_std_thresh(
    img: np.ndarray,
    thresh: float,
    cofactor: float = 5.0,
    kernel: int = 2,
    *,
    use_robust: bool = True,
    eps: float = 1e-8,
    return_mask: bool = False,
):
    """
    IMC marker preprocessing:
      1) arcsinh(cofactor * img)
      2) median filter (disk(kernel))
      3) z-score (mean/std) OR robust z (median/MAD)
      4) threshold: keep arcsinh values where score >= thresh else 0

    thresh is in z-score units (robust or classic depending on use_robust).
    """
    img = np.asarray(img, dtype=np.float32)

    if kernel < 0:
        raise ValueError("kernel must be >= 0")

    img_sin = np.arcsinh(img * float(cofactor))

    if kernel > 0:
        img_med = skimage.filters.median(img_sin, footprint=disk(int(kernel)))
    else:
        img_med = img_sin

    if use_robust:
        med = float(np.median(img_med))
        mad = float(np.median(np.abs(img_med - med)))
        scale = 1.4826 * mad
        z = (img_med - med) / max(scale, eps)
    else:
        mu = float(np.mean(img_med))
        sd = float(np.std(img_med))
        z = (img_med - mu) / max(sd, eps)

    out = np.where(z < float(thresh), 0.0, img_sin).astype(np.float32)

    if return_mask:
        return out, (z >= float(thresh))
    return out

## Execution
⚠️ USER INPUT REQUIRED  


In [ ]:
run_interactive_imc_preprocessing(
    path_raw=path_raw,
    path_img_preprocessed2_biopsies=path_img_preprocessed2_biopsies,
    path_img_preprocessed2_marker=path_img_preprocessed2_marker,
)

# 📊 **Images with multiple markers**
⚠️ USER INPUT REQUIRED  
Create a folder with the images, displaying several markers of different colors chosen for each biopsy.

In [ ]:
path_img=path+"images/"
path_raw=path+"images/raw/"
path_img_preprocessed2=path+"images/img_processing_2/"
path_img_preprocessed2_marker=path_img_preprocessed2+"Marker/"
path_img_preprocessed2_biopsies=path_img_preprocessed2+"Biopsies/"
path_img_preprocessed1=path+"images/img_processing_1/"
path_img_preprocessed1_marker=path_img_preprocessed1+"Marker/"
path_img_preprocessed1_biopsies=path_img_preprocessed1+"Biopsies/"

##### 🛠️ Functions


In [ ]:
SUPPORTED_EXT = (".tif", ".tiff", ".png", ".jpg", ".jpeg")

def read_gray(path: Path) -> np.ndarray:
    """Read image as float32 grayscale (H,W). If RGB, take channel 0."""
    if skio is not None:
        arr = skio.imread(str(path))
    else:
        arr = np.array(Image.open(path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr.astype(np.float32)


def find_marker_file(roi_dir: Path, marker: str) -> Optional[Path]:
    marker_low = marker.strip().lower()
    for p in roi_dir.iterdir():
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXT:
            if p.stem.strip().lower() == marker_low:
                return p
    return None



def to_u8_rgb(img01: np.ndarray) -> np.ndarray:
    return (np.clip(img01, 0, 1) * 255).astype(np.uint8)


def hex_to_rgb255(color_hex: str) -> Tuple[int, int, int]:
    c = str(color_hex).strip()
    if not c.startswith("#") or len(c) != 7:
        return (255, 255, 255)
    try:
        r = int(c[1:3], 16)
        g = int(c[3:5], 16)
        b = int(c[5:7], 16)
        return (r, g, b)
    except Exception:
        return (255, 255, 255)


def rgb255_to_rgb01(rgb: Tuple[int, int, int]) -> np.ndarray:
    return (np.array(rgb, dtype=np.float32) / 255.0).astype(np.float32)


def normalize_marker_to01(img: np.ndarray) -> np.ndarray:
    """
    Minimal mapping to [0,1] WITHOUT "processing":
    - If already 0..255 -> divide by 255
    - If already 0..1 -> keep
    - Else: clip to [0,255] then /255
    """
    x = img.astype(np.float32, copy=False)
    mx = float(np.nanmax(x)) if x.size else 0.0
    if mx <= 1.5:
        return np.clip(x, 0, 1).astype(np.float32)
    return (np.clip(x, 0, 255) / 255.0).astype(np.float32)



def _get_scalable_font(font_size: int) -> ImageFont.FreeTypeFont:
    font_size = int(font_size)
    try:
        return ImageFont.truetype("DejaVuSans.ttf", font_size)
    except Exception:
        pass
    try:
        from matplotlib import font_manager
        path = font_manager.findfont("DejaVu Sans", fallback_to_default=True)
        return ImageFont.truetype(path, font_size)
    except Exception:
        pass
    return ImageFont.load_default()


def annotate_composite(
    rgb_u8: np.ndarray,
    marker_to_rgb: Dict[str, Tuple[int, int, int]],
    font_size: int = 28,
    margin: int = 14,
    swatch_size: Tuple[int, int] = (26, 26),
    line_spacing: int = 8,
    background_alpha: int = 170,
    location: str = "topleft",
) -> np.ndarray:
    im = Image.fromarray(rgb_u8, mode="RGB").convert("RGBA")
    overlay = Image.new("RGBA", im.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    font = _get_scalable_font(font_size)

    items = [(m, marker_to_rgb[m]) for m in marker_to_rgb.keys()]
    if not items:
        return np.array(im.convert("RGB"))

    sw_w, sw_h = swatch_size

    text_w_max, text_h_max = 0, 0
    for m, _ in items:
        bbox = draw.textbbox((0, 0), str(m), font=font)
        tw = bbox[2] - bbox[0]
        th = bbox[3] - bbox[1]
        text_w_max = max(text_w_max, tw)
        text_h_max = max(text_h_max, th)

    line_h = max(sw_h, text_h_max)
    box_w = margin * 3 + sw_w + text_w_max
    box_h = margin * 2 + len(items) * line_h + (len(items) - 1) * line_spacing

    W, H = im.size
    if location == "topleft":
        x0, y0 = margin, margin
    elif location == "topright":
        x0, y0 = W - box_w - margin, margin
    elif location == "bottomleft":
        x0, y0 = margin, H - box_h - margin
    else:
        x0, y0 = W - box_w - margin, H - box_h - margin

    draw.rectangle([x0, y0, x0 + box_w, y0 + box_h], fill=(0, 0, 0, background_alpha))

    y = y0 + margin
    for m, (r, g, b) in items:
        draw.rectangle([x0 + margin, y, x0 + margin + sw_w, y + sw_h], fill=(r, g, b, 255))
        tx = x0 + margin * 2 + sw_w
        ty = y + (line_h - text_h_max) // 2
        draw.text((tx, ty), str(m), font=font, fill=(r, g, b, 255))
        y += line_h + line_spacing

    out = Image.alpha_composite(im, overlay).convert("RGB")
    return np.array(out, dtype=np.uint8)


# ======================================================
# Preview crop utilities
# ======================================================

def compute_crop_box(H: int, W: int, crop_h: int, crop_w: int) -> Tuple[int, int, int, int]:
    crop_h = int(max(1, min(crop_h, H)))
    crop_w = int(max(1, min(crop_w, W)))
    y0 = (H - crop_h) // 2
    x0 = (W - crop_w) // 2
    return y0, y0 + crop_h, x0, x0 + crop_w


# ======================================================
# Composite rendering (NO processing)
# ======================================================

def build_composite_rgb01(
    roi_dir: Path,
    selected_markers: List[str],
    marker_to_hex: Dict[str, str],
    marker_params: Dict[str, Dict],
    *,
    raw_cache: Optional[Dict[Tuple[str, str], np.ndarray]] = None,
    roi_name_for_cache: Optional[str] = None,
    downsample: int = 1,
    crop_box: Optional[Tuple[int, int, int, int]] = None,
    strict: bool = False,
) -> Tuple[np.ndarray, List[str]]:
    """
    Build composite as float RGB [0,1] for a single ROI directory.
    NO processing: each marker image is assumed already preprocessed.
    We only:
      - read (and optionally crop/downsample)
      - map to [0,1] (minimal: divide by 255 if needed)
      - multiply by chosen color and optional per-marker weight
      - accumulate
    """
    downsample = max(int(downsample), 1)

    acc = None
    H = W = None
    missing: List[str] = []

    for m in selected_markers:
        p = find_marker_file(roi_dir, m)
        if p is None:
            missing.append(m)
            if strict:
                raise FileNotFoundError(f"[{roi_dir.name}] Missing marker file: {m}.*")
            continue

        if raw_cache is not None and roi_name_for_cache is not None:
            key = (roi_name_for_cache, m)
            if key not in raw_cache:
                raw_cache[key] = read_gray(p)
            img = raw_cache[key]
        else:
            img = read_gray(p)

        if crop_box is not None:
            y0, y1, x0, x1 = crop_box
            img = img[y0:y1, x0:x1]

        if downsample > 1:
            img = img[::downsample, ::downsample]

        if acc is None:
            H, W = img.shape[:2]
            acc = np.zeros((H, W, 3), dtype=np.float32)
        else:
            if img.shape[:2] != (H, W):
                missing.append(m)
                if strict:
                    raise ValueError(f"[{roi_dir.name}] Size mismatch for marker {m}: {img.shape[:2]} vs {(H, W)}")
                continue

        img01 = normalize_marker_to01(img)

        rgb255 = hex_to_rgb255(marker_to_hex.get(m, "#ffffff"))
        color01 = rgb255_to_rgb01(rgb255)

        w = float(marker_params.get(m, {}).get("weight", 1.0))
        acc += (img01[..., None] * color01[None, None, :]) * w

    if acc is None:
        acc = np.zeros((1, 1, 3), dtype=np.float32)

    return acc.astype(np.float32), missing


def normalize_composite(acc01: np.ndarray, mode: str = "clip") -> np.ndarray:
    mode = str(mode).lower()
    if mode == "rescale":
        mx = float(acc01.max())
        if mx > 1.0:
            return (acc01 / mx).astype(np.float32)
    return np.clip(acc01, 0, 1).astype(np.float32)


def generate_composite_images(
    path_raw: str | Path,
    path_out_root: str | Path,
    selected_markers: List[str],
    marker_to_hex: Dict[str, str],
    marker_params: Dict[str, Dict],
    *,
    strict: bool = False,
    composite_mode: str = "clip",
    add_legend: bool = True,
    legend_font: int = 28,
    legend_location: str = "topleft",
) -> str:
    path_raw = Path(path_raw)
    path_out_root = Path(path_out_root)

    safe_name = "_".join([m.replace(" ", "_") for m in selected_markers])
    out_dir = path_out_root / f"composite_{safe_name}"
    out_dir.mkdir(parents=True, exist_ok=True)

    rois = sorted([p for p in path_raw.iterdir() if p.is_dir()])

    for roi_dir in tqdm(rois, desc="Composites", unit="ROI"):
        acc01, missing = build_composite_rgb01(
            roi_dir=roi_dir,
            selected_markers=selected_markers,
            marker_to_hex=marker_to_hex,
            marker_params=marker_params,
            downsample=1,
            crop_box=None,  # FULL
            raw_cache=None,
            roi_name_for_cache=None,
            strict=strict,
        )
        acc01 = normalize_composite(acc01, mode=composite_mode)
        rgb_u8 = to_u8_rgb(acc01)

        if add_legend:
            marker_to_rgb = {
                m: hex_to_rgb255(marker_to_hex.get(m, "#ffffff"))
                for m in selected_markers
                if m not in missing
            }
            rgb_u8 = annotate_composite(
                rgb_u8,
                marker_to_rgb=marker_to_rgb,
                font_size=int(legend_font),
                location=str(legend_location),
            )

        out_path = out_dir / f"{roi_dir.name}.png"
        Image.fromarray(rgb_u8).save(out_path)

        if missing:
            tqdm.write(f"[{roi_dir.name}] missing skipped: {', '.join(missing)}")

    print(f"✅ Composites created in: {out_dir}")
    return str(out_dir)



def make_float_input(value: float, step: float, description: str, width: str = "380px") -> widgets.BoundedFloatText:
    return widgets.BoundedFloatText(
        value=float(value),
        min=-1e18,
        max=1e18,
        step=float(step),
        description=description,
        layout=widgets.Layout(width=width),
        style={"description_width": "70px"}
    )

def make_bool_input(value: bool, description: str, width: str = "180px") -> widgets.Checkbox:
    return widgets.Checkbox(value=bool(value), description=description, indent=False, layout=widgets.Layout(width=width))


# ======================================================
# Interactive UI (colors + weights + clean preview + preview crop)
# ======================================================

@dataclass
class MarkerDefault:
    weight: float = 1.0


def interactive_composite_builder(
    path_raw: str | Path,
    list_all_markers: List[str],
    path_out_root: str | Path,
    *,
    title: str = "🎛️ Composite builder (NO processing: just combine preprocessed markers)",
    defaults: Optional[MarkerDefault] = None,
) -> Dict:
    defaults = defaults or MarkerDefault()
    path_raw = Path(path_raw)
    path_out_root = Path(path_out_root)

    rois = sorted([p.name for p in path_raw.iterdir() if p.is_dir()])
    list_all_markers = sorted({str(m).strip() for m in list_all_markers if str(m).strip()})

    state = {"selected_markers": [], "marker_to_hex": {}, "marker_params": {}}

    header = widgets.HTML(f"<h3 style='margin:0 0 10px 0;'>{title}</h3>")

    marker_selector = widgets.SelectMultiple(
        options=list_all_markers,
        description="Markers:",
        rows=min(16, len(list_all_markers)),
        layout=widgets.Layout(width="520px", height="260px")
    )

    btn_build = widgets.Button(description="🧩 Build panels", button_style="info")
    btn_confirm = widgets.Button(description="✅ Confirm", button_style="success")
    btn_export = widgets.Button(description="💾 Export all ROIs", button_style="warning")

    roi_cb = widgets.Combobox(
        placeholder="Type ROI name…",
        options=rois,
        description="Preview ROI:",
        ensure_option=True,
        layout=widgets.Layout(width="520px"),
        style={"description_width": "120px"}
    )

    composite_mode_dd = widgets.Dropdown(
        options=["clip", "rescale"],
        value="clip",
        description="Composite mode:",
        layout=widgets.Layout(width="520px"),
        style={"description_width": "120px"}
    )

    crop_enable_cb = widgets.Checkbox(value=True, description="Preview crop (center)", indent=False, layout=widgets.Layout(width="220px"))
    crop_h_in = widgets.BoundedIntText(value=1200, min=128, max=10000, step=64, description="Crop H:",
                                      layout=widgets.Layout(width="250px"), style={"description_width": "70px"})
    crop_w_in = widgets.BoundedIntText(value=1200, min=128, max=10000, step=64, description="Crop W:",
                                      layout=widgets.Layout(width="250px"), style={"description_width": "70px"})
    downsample_in = widgets.BoundedIntText(value=1, min=1, max=16, step=1, description="Downsample:",
                                          layout=widgets.Layout(width="520px"), style={"description_width": "120px"})

    add_legend_cb = widgets.Checkbox(value=True, description="Add legend on export", indent=False, layout=widgets.Layout(width="220px"))
    legend_font_in = widgets.BoundedIntText(value=28, min=8, max=120, step=1, description="Legend font:",
                                           layout=widgets.Layout(width="520px"), style={"description_width": "120px"})
    legend_loc_dd = widgets.Dropdown(options=["topleft", "topright", "bottomleft", "bottomright"], value="topleft",
                                     description="Legend loc:", layout=widgets.Layout(width="520px"),
                                     style={"description_width": "120px"})

    strict_cb = widgets.Checkbox(value=False, description="Strict (error if missing)", indent=False, layout=widgets.Layout(width="220px"))

    colors_box = widgets.VBox([])
    params_box = widgets.VBox([])
    preview_out = widgets.Output()
    log_out = widgets.Output()

    default_palette = [
        "#e60000", "#00f504", "#0068b3", "#fff700", "#861f93", "#ffa200",
        "#00ffff", "#ff00c8", "#1000f0", "#f6fefd", "#03e212", "#aaaaaa"
    ]
    color_pickers: Dict[str, widgets.ColorPicker] = {}
    param_widgets: Dict[str, Dict[str, widgets.Widget]] = {}
    raw_cache: Dict[Tuple[str, str], np.ndarray] = {}

    def _default_params() -> Dict:
        return {"weight": defaults.weight}

    def _make_color_row(marker: str, default_hex: str) -> widgets.HBox:
        name = widgets.HTML(f"<b style='font-size:14px'>{marker}</b>", layout=widgets.Layout(width="220px"))
        cp = widgets.ColorPicker(value=default_hex, concise=False, layout=widgets.Layout(width="220px"))
        color_pickers[marker] = cp
        return widgets.HBox([name, cp], layout=widgets.Layout(gap="12px", align_items="center"))

    def _make_params_panel(marker: str, p0: Dict) -> widgets.VBox:
        title_html = widgets.HTML(f"<b style='font-size:14px'>{marker}</b>", layout=widgets.Layout(width="220px"))
        weight = make_float_input(p0.get("weight", defaults.weight), step=0.05, description="weight")
        param_widgets[marker] = {"weight": weight}
        row = widgets.HBox([title_html, weight])
        return widgets.VBox([row], layout=widgets.Layout(border="1px solid #ddd", padding="8px"))

    def _current_marker_params(m: str) -> Dict:
        if m not in param_widgets:
            return _default_params()
        return {"weight": float(param_widgets[m]["weight"].value)}

    def _collect_live_settings():
        sels = list(marker_selector.value)
        marker_to_hex_live = {m: color_pickers[m].value for m in sels if m in color_pickers}
        marker_params_live = {m: _current_marker_params(m) for m in sels}
        return sels, marker_to_hex_live, marker_params_live

    def _render_preview():
        with preview_out:
            preview_out.clear_output(wait=True)

            sels, marker_to_hex_live, marker_params_live = _collect_live_settings()
            if not sels:
                return

            roi = roi_cb.value
            if not roi:
                return
            roi_dir = path_raw / roi
            if not roi_dir.exists():
                return

            crop_box = None
            if bool(crop_enable_cb.value):
                p0 = find_marker_file(roi_dir, sels[0])
                if p0 is not None:
                    key_shape = (roi, "__shape__")
                    if key_shape not in raw_cache:
                        raw_cache[key_shape] = read_gray(p0)
                    img0 = raw_cache[key_shape]
                    H0, W0 = img0.shape[:2]
                    crop_box = compute_crop_box(H0, W0, int(crop_h_in.value), int(crop_w_in.value))

            acc01, _ = build_composite_rgb01(
                roi_dir=roi_dir,
                selected_markers=sels,
                marker_to_hex=marker_to_hex_live,
                marker_params=marker_params_live,
                downsample=max(int(downsample_in.value), 1),
                crop_box=crop_box,
                raw_cache=raw_cache,
                roi_name_for_cache=roi,
                strict=False,
            )
            acc01 = normalize_composite(acc01, mode=str(composite_mode_dd.value))
            rgb_u8 = to_u8_rgb(acc01)

            fig = plt.figure(figsize=(7.2, 7.2), frameon=False)
            ax = plt.Axes(fig, [0.0, 0.0, 1.0, 1.0])
            ax.set_axis_off()
            fig.add_axes(ax)
            ax.imshow(rgb_u8)
            plt.show()

    def _attach_live_observers():
        for _, ww in param_widgets.items():
            for _, w in ww.items():
                w.observe(lambda change: _render_preview(), names="value")
        for _, cp in color_pickers.items():
            cp.observe(lambda change: _render_preview(), names="value")

    def on_build(_):
        sels = list(marker_selector.value)
        with log_out:
            log_out.clear_output(wait=True)
            if not sels:
                print("Select at least one marker first.")
                colors_box.children = []
                params_box.children = []
                color_pickers.clear()
                param_widgets.clear()
                return

            color_pickers.clear()
            color_rows = []
            for i, m in enumerate(sels):
                c0 = state["marker_to_hex"].get(m, default_palette[i % len(default_palette)])
                color_rows.append(_make_color_row(m, c0))
            colors_box.children = color_rows

            param_widgets.clear()
            params_panels = []
            for m in sels:
                p0 = state["marker_params"].get(m, _default_params())
                params_panels.append(_make_params_panel(m, p0))
            params_box.children = params_panels

            _attach_live_observers()
            print(f"Panels built for {len(sels)} marker(s). Preview updates live on any change.")
        _render_preview()

    def on_confirm(_):
        sels, marker_to_hex_live, marker_params_live = _collect_live_settings()
        state["selected_markers"] = sels
        state["marker_to_hex"] = marker_to_hex_live
        state["marker_params"] = marker_params_live

        with log_out:
            log_out.clear_output(wait=True)
            print("✔ Selected markers:", state["selected_markers"])
            print("✔ marker_to_hex:")
            print(state["marker_to_hex"])
            print("✔ marker_params (per marker):")
            for m in state["selected_markers"]:
                print(f"  - {m}: {state['marker_params'][m]}")

        _render_preview()

    def on_export(_):
        on_confirm(None)
        if not state["selected_markers"]:
            with log_out:
                print("Nothing to export (no markers selected).")
            return

        out_dir = generate_composite_images(
            path_raw=path_raw,
            path_out_root=path_out_root,
            selected_markers=state["selected_markers"],
            marker_to_hex=state["marker_to_hex"],
            marker_params=state["marker_params"],
            strict=bool(strict_cb.value),
            composite_mode=str(composite_mode_dd.value),
            add_legend=bool(add_legend_cb.value),
            legend_font=int(legend_font_in.value),
            legend_location=str(legend_loc_dd.value),
        )
        with log_out:
            print(f"✅ Export done: {out_dir}")

    btn_build.on_click(on_build)
    btn_confirm.on_click(on_confirm)
    btn_export.on_click(on_export)

    roi_cb.observe(lambda change: _render_preview(), names="value")
    composite_mode_dd.observe(lambda change: _render_preview(), names="value")
    downsample_in.observe(lambda change: _render_preview(), names="value")
    crop_enable_cb.observe(lambda change: _render_preview(), names="value")
    crop_h_in.observe(lambda change: _render_preview(), names="value")
    crop_w_in.observe(lambda change: _render_preview(), names="value")

    left = widgets.VBox([
        header,
        marker_selector,
        widgets.HBox([btn_build, btn_confirm, btn_export]),
        widgets.HTML("<hr><b>Colors</b>"),
        colors_box,
        widgets.HTML("<hr><b>Per-marker settings (NO processing)</b>"),
        params_box,
        widgets.HTML("<hr><b>Export (legend on saved images)</b>"),
        add_legend_cb,
        legend_font_in,
        legend_loc_dd,
        log_out
    ], layout=widgets.Layout(width="560px"))

    right = widgets.VBox([
        widgets.HTML("<b>Preview (clean image only)</b>"),
        roi_cb,
        composite_mode_dd,
        widgets.HTML("<b>Preview crop (center portion for huge biopsies)</b>"),
        widgets.HBox([crop_enable_cb, crop_h_in, crop_w_in], layout=widgets.Layout(gap="10px")),
        downsample_in,
        widgets.HBox([strict_cb], layout=widgets.Layout(gap="16px")),
        preview_out
    ], layout=widgets.Layout(width="720px"))

    display(widgets.HBox([left, right], layout=widgets.Layout(gap="18px")))
    return state

##### ⚙️ Execution
⚠️ USER INPUT REQUIRED  

In [ ]:

preprocessing_type=input("Enter 1 or 2 to choose the preprocessing type you prefer : ")

Enter 1 or 2 to choose the preprocessing type you prefer : 2


In [ ]:

roi0 = sorted([p.name for p in Path(path_raw).iterdir() if p.is_dir()])[0]
list_all_markers = sorted({Path(f).stem for f in os.listdir(Path(path_raw) / roi0)})

state = interactive_composite_builder(

     path_raw=path+"images/img_processing_"+preprocessing_type+"/Biopsies/",
     list_all_markers=list_all_markers,
     path_out_root=path_img
 )


Composites:   0%|          | 0/89 [00:00<?, ?ROI/s]/tmp/ipykernel_6416/2601755613.py:85: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  im = Image.fromarray(rgb_u8, mode="RGB").convert("RGBA")
Composites:   1%|          | 1/89 [00:01<02:29,  1.70s/ROI]/tmp/ipykernel_6416/2601755613.py:85: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  im = Image.fromarray(rgb_u8, mode="RGB").convert("RGBA")
Composites:   2%|▏         | 2/89 [00:03<02:14,  1.54s/ROI]/tmp/ipykernel_6416/2601755613.py:85: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  im = Image.fromarray(rgb_u8, mode="RGB").convert("RGBA")
Composites:   3%|▎         | 3/89 [00:05<02:35,  1.81s/ROI]/tmp/ipykernel_6416/2601755613.py:85: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  im = Image.fromarray(rgb_u8, mode="RGB").convert("RGB

✅ Composites created in: /content/gdrive/MyDrive/these/pipeline/rejection/images/composite_Aquaporin1_CD138_ColT1_ColT4_Vimentin
